In [25]:
import pangolin
from pangolin import interface as pi
from pangolin import ir
RV = pangolin.ir.RV

In [26]:
def autobatch(vars: list[pangolin.RV]) -> dict[RV,RV]:
    pass


In [40]:
def get_merge_constants_data():
    xs = [pi.constant(2*i) for i in range(5)]

    xs_vmapped = pi.constant([2*i for i in range(5)])
    new_xs = [xs_vmapped[i] for i in range(5)]

    d = {xs[i] : new_xs[i] for i in range(5)}
    return xs, d

def test_merge_constants():
    xs, d = get_merge_constants_data()
    
    #d = autobatch(xs) # uncomment this

    # check each parent is an index into constant vector
    for n in range(5):
        old = xs[n]
        new = d[old]
        assert new.op == ir.Index()
        a, b = new.parents
        assert a.op == ir.Constant([0, 2, 4, 6, 8])
        assert b.op == ir.Constant(n)

    a = d[xs[0]].parents[0]
    
    # check constant vector is the same
    for old in xs:
        new = d[old]
        assert new.parents[0] == a

test_merge_constants()

# start
# xs = [constant(0.0),constant(1.1),constant(2.2),constant(3.3),constant(4.4)]
# end
# a = constant([0.0, 1.1, 2.2, 3.3, 4.4])
# new_xs = [a[0], a[1], a[2], a[3], a[4]]

In [45]:
def get_merge_constants_downstream_data():
    xs = [pi.constant(2*i) for i in range(5)]
    ys = [x + 1 for x in xs]
    
    xs_vmapped = pi.constant([2*i for i in range(5)])
    new_xs = [xs_vmapped[i] for i in range(5)]

    ys_vmapped = pi.vmap(pi.add, [0,None])(xs_vmapped, pi.constant(1))
    new_ys = [ys_vmapped[i] for i in range(5)]
    
    d = {xs[i] : new_xs[i] for i in range(5)} | {ys[i] : new_ys[i] for i in range(5)}
    
    return (xs, ys), d


def test_merge_constants_downstream():
    (xs, ys), d = get_merge_constants_downstream_data()
    
    #d = autobatch(ys) # yes! pass ys, not xs!
    
    # check each parent is an index into constant vector
    for n in range(5):
        old = xs[n]
        new = d[old]
        a, b = new.parents
        assert new.op == ir.Index()
        assert a.op == ir.Constant([0, 2, 4, 6, 8])
        assert b.op == ir.Constant(n)

    a = d[xs[0]].parents[0]
    
    # check constant vector is the same
    for old in xs:
        new = d[old]
        assert new.parents[0] == a

    # check each parent is an index into a
    for n in range(5):
        old = ys[n]
        new = d[old]
        c, f = new.parents
        assert new.op == ir.Index()
        assert f.op == ir.Constant(n)

    c == d[ys[0]].parents[0]
    for old in ys:
        new = d[old]
        assert new.parents[0] == c

    assert c.op == ir.VMap(ir.Add(), (0, None), 5)

test_merge_constants_downstream()
        
# start
# xs = [constant(0.0),constant(1.1),constant(2.2),constant(3.3),constant(4.4)]
# ys = [xs[0]+1, xs[1]+1, ..., xs[4]+1]
# end
# a = constant([0.0, 1.1, 2.2, 3.3, 4.4])
# new_xs = [a[0], a[1], a[2], a[3], a[4]]
# c = vmap(add,[0,None])(a, constant(1))
# new_ys = [e[0], e[1], e[2], e[3], e[4]]

In [48]:
m = pi.constant(0)
s = pi.constant(1)
xs = [pi.normal(m,s) for _ in range(5)]
ir.graph_upstream(xs)

○       () 0
│
│ ○     () 1
│ │
├─┼─○   () normal
│ │
├─┼─○   () normal
│ │
├─┼─○   () normal
│ │
├─┼─○   () normal
│ │
└─○     () normal


In [ ]:
xs = [pi.normal(0,1) for _ in range(5)]
ir.graph_upstream(xs)

# correct output somewhat debatable (would accept either of these)

# vmap(normal,[0,0])([0,0,0,0,0], [1,1,1,1,1])
# vmap(normal,[None,None], 5)(0, 1) # would be better

○                     () 0
│
│ ○                   () 1
│ │
│ │ ○                 () 0
│ │ │
│ │ │ ○               () 1
│ │ │ │
│ │ │ │ ○             () 0
│ │ │ │ │
│ │ │ │ │ ○           () 1
│ │ │ │ │ │
│ │ │ │ │ │ ○         () 0
│ │ │ │ │ │ │
│ │ │ │ │ │ │ ○       () 1
│ │ │ │ │ │ │ │
│ │ │ │ │ │ │ │ ○     () 0
│ │ │ │ │ │ │ │ │
│ │ │ │ │ │ │ │ │ ○   () 1
│ │ │ │ │ │ │ │ │ │
└─○ │ │ │ │ │ │ │ │   () normal
    │ │ │ │ │ │ │ │
    └─○ │ │ │ │ │ │   () normal
        │ │ │ │ │ │
        └─○ │ │ │ │   () normal
            │ │ │ │
            └─○ │ │   () normal
                │ │
                └─○   () normal


In [ ]:
xs = [pi.constant(i) for i in range(5)]
ys = [x + 3 for x in [xs[3], xs[0], xs[4], xs[1], xs[2]]]

# should yield:
# xs_new = pi.constant([0,1,2,3,4])
# tmp = xs_new[3,0,4,1,2]
# ys_new = pi.vmap(pi.add,[0,None])(tmp, 3)

In [ ]:
xs = [pi.constant(i) for i in range(5)]
ys = [pi.constant(i) for i in range(5)]
zs = [x + y for x, y in zip([xs[3], xs[0], xs[4], xs[1], xs[2]],
                            [ys[1], ys[0], ys[2], ys[4], ys[3]])]
ws = [x + y for x, y in zip([xs[2], xs[0], xs[4], xs[1], xs[3]],
                            [ys[4], ys[0], ys[2], ys[1], ys[3]])]


# should yield:
# something more complicated...